# Prototype: Hệ thống xếp ca làm việc (MVP – Solver only)

**Phạm vi MVP (đã chốt qua Q&A):**
- Chỉ tập trung vào **lõi solver OR-Tools** (CP-SAT) — input là dữ liệu mẫu hardcode trong notebook, output là bảng lịch (chưa có backend Golang / frontend React / DB thật).
- Horizon: **1 tháng (4 tuần / 28 ngày)**, có cơ chế bù trừ effort giữa các tuần.
- Có mô phỏng **nghỉ phép** (không ép bù đủ ca).
- Có demo **giữ lịch đã approve, đổi ca cục bộ** khi có sự cố đột xuất (không ghi đè toàn bộ lịch đã duyệt).

**Các nghiệp vụ được đưa vào solver (theo lựa chọn của bạn):**
1. Min-staffing theo cổng (A/B/G/D) × ca (sáng/đêm) × vai trò — định nghĩa bằng **data/config**, không hardcode logic.
2. Phân cấp vai trò: Trưởng ca (TC) chỉ làm cổng B; Phó ca (PC) có thể thay TC ở slot lead khi cần.
3. Không xếp ca sáng ngay sau ca đêm hôm trước.
4. Cân bằng effort giữa nhân viên (theo target 44h/tuần, quy đổi ra giờ theo thời lượng ca thật) + ưu tiên xen kẽ sáng/đêm (phạt streak 3 ngày liên tiếp cùng loại ca).
5. Nghỉ phép: nhân viên nghỉ sẽ được **giảm target theo tỷ lệ ngày available**, không bị ép bù.
6. Giữ lịch đã approve khi re-solve (biến bị khóa = hard constraint), chỉ mở lại đúng phần bị ảnh hưởng.
7. **Thời lượng mỗi ca khác nhau theo từng cổng** (`SHIFT_HOURS`) — effort/target tính đúng theo GIỜ thay vì đếm số ca, đúng mô tả gốc "khối lượng công việc khác nhau, thời gian mỗi ca làm khác nhau".

**Không nằm trong MVP lần này** (để các vòng sau): giao diện web thật, DB, workflow approve nhiều bước qua UI, thông báo/notification, xử lý đồng thời nhiều yêu cầu đổi ca cùng lúc.

> Chạy tuần tự từng cell từ trên xuống. Cell đầu tiên cài `ortools` (mất khoảng 20-30s trên Colab).

In [ ]:
!pip install ortools -q


## 1. Config nghiệp vụ (data, không hardcode logic)

Đây là phần **quan trọng nhất để dễ chỉnh sửa sau này**: toàn bộ số lượng yêu cầu nhân sự theo cổng/ca, **thời lượng mỗi ca theo từng cổng**, cũng như các trọng số ưu tiên, đều để dưới dạng biến/dict — đúng theo yêu cầu "vì số lượng và vai trò có thể thay đổi tùy tình hình thực tế, nên không thể hard-code cứng", và đúng mô tả gốc: *"4 cổng A, B, G, D có khối lượng công việc khác nhau, **thời gian mỗi ca làm khác nhau**"*.

In [ ]:
import random
from dataclasses import dataclass, field
from datetime import date, timedelta
from ortools.sat.python import cp_model
import pandas as pd

random.seed(42)

# ---- Cổng & loại ca ----
GATES = ["A", "B", "G", "D"]
SHIFT_TYPES = ["sang", "dem"]  # sang = ca sáng, dem = ca đêm

# ---- Horizon lịch: 1 tháng = 4 tuần = 28 ngày ----
NUM_WEEKS = 4
NUM_DAYS = NUM_WEEKS * 7
START_DATE = date(2026, 9, 7)  # thứ Hai tuần đầu tiên
DAYS = [START_DATE + timedelta(days=i) for i in range(NUM_DAYS)]

# ---- Vai trò ----
ROLE_NV, ROLE_TC, ROLE_PC = "NV", "TC", "PC"  # nhân viên / trưởng ca / phó ca

# ---- Yêu cầu nhân sự tối thiểu theo cổng/ca (DATA — sửa ở đây, không đụng code solver) ----
# "LEAD" = slot cần trưởng ca (hoặc phó ca thay thế tùy lead_mandatory_role)
REQUIREMENTS = {
    "A": {"sang": {"NV": 1},                "dem": {"NV": 1}},
    "B": {"sang": {"NV": 2, "LEAD": 1, "lead_mandatory_role": False},   # ca sáng: PC được thay TC
          "dem":  {"NV": 1, "LEAD": 1, "lead_mandatory_role": True}},  # ca đêm: bắt buộc đúng TC
    "G": {"sang": {"NV": 1},                "dem": {"NV": 1}},
    "D": {"sang": {"NV": 1},                "dem": {"NV": 2}},
}
LEAD_GATES = {"B"}  # cổng nào có khái niệm "lead slot" (trưởng ca chỉ làm ở đây)

# ---- Thời lượng mỗi ca theo từng cổng (SỐ THẬT do bạn cung cấp, từ bảng phân ca thực tế) ----
# Đây chính là phần "thời gian mỗi ca làm khác nhau" — dùng để tính effort theo GIỜ
# thay vì đếm số ca (đếm số ca sẽ sai nếu ca cổng này dài/ngắn khác cổng kia).
# Cổng A/G: ca đêm dài hơn ca sáng 2h (13h vs 11h). Cổng B/D: ca đêm dài hơn ca sáng 1h (12h vs 11h).
SHIFT_HOURS = {
    "A": {"sang": 11, "dem": 13},
    "B": {"sang": 11, "dem": 12},
    "G": {"sang": 11, "dem": 13},
    "D": {"sang": 11, "dem": 12},
}

# ---- Mục tiêu effort & trọng số phạt trong objective ----
BASE_TARGET_HOURS_PER_WEEK = 44     # ĐÚNG đơn vị nghiệp vụ: 44 giờ/TUẦN (không phải /tháng)
BASE_TARGET_HOURS_PER_MONTH = BASE_TARGET_HOURS_PER_WEEK * NUM_WEEKS  # quy đổi cho horizon 4 tuần hiện tại
SHORTFALL_PENALTY = 1000           # phạt nặng khi thiếu người -> ưu tiên tránh trước
LEAD_SHORTFALL_PENALTY = 800       # phạt khi thiếu trưởng ca/phó ca
BALANCE_PENALTY_WEIGHT = 1         # phạt lệch effort giữa nhân viên (đơn vị GIỜ, nên nhỏ hơn trước)
STREAK_PENALTY_WEIGHT = 2          # phạt xếp 3 ngày liên tiếp cùng loại ca (khuyến khích xen kẽ)

print(f"Horizon: {NUM_DAYS} ngày, từ {DAYS[0]} đến {DAYS[-1]}")


### 1b. Kiểm tra năng lực (capacity check) — TRƯỚC khi solve

Trước khi chạy solver, nên kiểm tra nhanh: **tổng nhu cầu giờ công tối thiểu/tháng** (suy ra từ `REQUIREMENTS` × `SHIFT_HOURS`) so với **tổng khả năng cung ứng nếu mỗi nhân viên chỉ làm đúng target 44h/tuần**. Nếu cung < cầu, solver vẫn sẽ ra lịch (ưu tiên đủ người hơn đúng target — theo đúng thứ tự ưu tiên đã thống nhất), nhưng nhiều nhân viên sẽ vượt target, và đó là tín hiệu cần bổ sung nhân sự chứ không phải lỗi solver.

> **Nguồn số liệu:** `SHIFT_HOURS` ở trên lấy từ bảng phân ca tuần thực tế công ty đang dùng (ảnh chụp Excel) — không còn là số ước lượng. Vài ghi chú đối chiếu từ bảng thật, hữu ích khi thiết kế tầng dữ liệu/API ở bước tiếp theo (chưa cần đổi solver ngay):
> - Mã "N" trong bảng thật = nhân viên **tự đăng ký không làm** hôm đó (không phải 1 loại ca) → tương đương `availability=False` cả 2 ca ngày đó trong model hiện tại.
> - Ô tô màu xanh trong bảng thật đánh dấu ngày bị chặn do **đã làm đêm cuối tuần trước** — xác nhận đúng nhu cầu rolling-horizon nối tiếp giữa các tuần (đã ghi nhận là concern cần chốt ở tầng kiến trúc, xem trao đổi trước).
> - "P"/"NS" (phép / nghỉ thai sản) đều xử lý như nhau qua `leave_days`, đúng như đã thống nhất.
> - Nhóm nhân viên ca hành chính cố định "8" trong bảng thật **không** thuộc phạm vi bài toán này.

In [ ]:
demand_per_day = sum(
    (REQUIREMENTS[g][s].get("NV", 0) + REQUIREMENTS[g][s].get("LEAD", 0)) * SHIFT_HOURS[g][s]
    for g in GATES for s in SHIFT_TYPES
)
demand_per_month = demand_per_day * NUM_DAYS
n_employees_needed = demand_per_month / BASE_TARGET_HOURS_PER_MONTH

print(f"Tổng nhu cầu tối thiểu: {demand_per_day} giờ-người/ngày -> {demand_per_month} giờ-người/tháng")
print(f"Với target {BASE_TARGET_HOURS_PER_WEEK}h/người/TUẦN (~{BASE_TARGET_HOURS_PER_MONTH}h/tháng cho horizon {NUM_WEEKS} tuần), "
      f"cần tối thiểu ~{n_employees_needed:.0f} nhân viên để KHÔNG ai phải vượt target "
      f"(chưa kể availability < 100%, nên thực tế cần nhỉnh hơn số này).")


## 2. Dữ liệu mẫu: nhân viên, đăng ký ca hàng tuần, nghỉ phép

- `Employee`: id, tên, vai trò (NV/TC/PC), và `leave_days` (các ngày nghỉ phép/nghỉ chế độ — nếu nghỉ thì **không bị ép bù ca**, target sẽ tự giảm theo tỉ lệ).
- `make_sample_availability`: mô phỏng việc **nhân viên đăng ký thời gian có thể đi làm hàng tuần** (đăng ký lặp lại theo tuần, có chút ngẫu nhiên để giống thực tế). Đây chính là input mà trong hệ thống thật nhân viên sẽ nhập qua UI.

> Bộ dữ liệu mẫu bên dưới có 21 nhân viên — theo capacity check ở mục 1b (dùng giờ ca THẬT), cần ~22-23 người để không ai vượt target; 21 người là khá sát nên actual hours sẽ nhỉnh hơn target một chút (xem mục 5).

In [ ]:
@dataclass
class Employee:
    eid: str
    name: str
    role: str                      # NV / TC / PC
    leave_days: set = field(default_factory=set)  # index ngày (0..NUM_DAYS-1) nghỉ phép


def make_sample_employees():
    names_nv = [f"NV{i:02d}" for i in range(1, 17)]   # 16 nhân viên thường
    names_tc = ["TC01", "TC02", "TC03"]                # 3 trưởng ca (chỉ làm cổng B)
    names_pc = ["PC01", "PC02"]                          # 2 phó ca (backup trưởng ca)
    emps = [Employee(n, n, ROLE_NV) for n in names_nv]
    emps += [Employee(n, n, ROLE_TC) for n in names_tc]
    emps += [Employee(n, n, ROLE_PC) for n in names_pc]

    # Mô phỏng nghỉ phép: NV05 nghỉ chế độ 5 ngày liên tục (tuần 2), NV09 nghỉ phép 2 ngày
    for e in emps:
        if e.eid == "NV05":
            e.leave_days = set(range(7, 12))
        if e.eid == "NV09":
            e.leave_days = set(range(20, 22))
    return emps


def make_sample_availability(employees):
    """Mô phỏng đăng ký ca hàng tuần: mỗi nhân viên có 1 pattern theo 7 ngày trong
    tuần (lặp lại các tuần), ngày nghỉ phép thì tự động không available."""
    avail = {}
    for e in employees:
        pattern = {}
        for wd in range(7):
            can_sang = random.random() < 0.72
            can_dem = random.random() < 0.62
            if not can_sang and not can_dem:
                can_sang = True  # tránh trường hợp hoàn toàn không đăng ký ngày nào
            pattern[wd] = {"sang": can_sang, "dem": can_dem}
        day_avail = {}
        for d in range(NUM_DAYS):
            if d in e.leave_days:
                day_avail[d] = {"sang": False, "dem": False}
            else:
                day_avail[d] = dict(pattern[d % 7])
        avail[e.eid] = day_avail
    return avail


employees = make_sample_employees()
availability = make_sample_availability(employees)
print(f"Tổng số nhân viên: {len(employees)}  (NV={sum(e.role=='NV' for e in employees)}, "
      f"TC={sum(e.role=='TC' for e in employees)}, PC={sum(e.role=='PC' for e in employees)})")
print(f"Nhân viên có nghỉ phép trong tháng: {[e.eid for e in employees if e.leave_days]}")


## 3. Solver (Google OR-Tools CP-SAT)

Mapping từng nghiệp vụ → constraint trong model:

| # | Nghiệp vụ | Cách hiện thực |
|---|---|---|
| a | Mỗi nhân viên tối đa 1 ca/ngày, chỉ được xếp nếu đã đăng ký available | `sum(...) <= 1` + ép biến = 0 nếu không available |
| b | Trưởng ca (TC) chỉ làm cổng B | ép biến = 0 với mọi cổng khác B |
| c | Không xếp sáng ngay sau đêm hôm trước | `night[d] + morning[d+1] <= 1` |
| d | Giữ lịch đã approve khi re-solve | các assignment trong dict `locked` bị ép bằng giá trị cố định |
| e | Min-staffing theo cổng/ca + vai trò lead, cho phép thiếu nhưng **phạt nặng + gắn cờ manual review** | biến slack `short` / `lead_short`, cộng vào objective với trọng số lớn |
| f | Cân bằng effort **44h/TUẦN** (tính đúng theo GIỜ, có xét thời lượng ca khác nhau theo cổng), nghỉ phép không bị ép bù | target giờ = `BASE_TARGET_HOURS_PER_WEEK × số tuần × (ngày available / tổng ngày)`; effort thực tế = tổng `SHIFT_HOURS[gate][shift]` của các ca được xếp; phạt độ lệch giờ, cân bằng ở mức tổng cả horizon (chính là cơ chế bù trừ giữa các tuần) |
| g | Ưu tiên xen kẽ sáng/đêm | phạt khi có streak 3 ngày liên tiếp cùng loại ca |

Toàn bộ các ràng buộc "thiếu người" đều dùng **slack variable** thay vì infeasible cứng — đúng tinh thần tài liệu: *"nếu không thỏa có thể require manual review từ quản lý"* thay vì hệ thống bị treo/không giải được.

In [ ]:
def solve_schedule(employees, availability, locked=None, time_limit_s=20):
    """
    locked: dict {(eid, day_index): (gate, shift) | "OFF"} — các assignment đã được
            quản lý approve, solver PHẢI giữ nguyên (dùng khi re-solve cục bộ).
    Trả về: (status, schedule_df, shortage_df, stats)
    """
    locked = locked or {}
    model = cp_model.CpModel()
    eids = [e.eid for e in employees]
    emp_by_id = {e.eid: e for e in employees}

    assign = {}
    for e in eids:
        for d in range(NUM_DAYS):
            for g in GATES:
                for s in SHIFT_TYPES:
                    assign[e, d, g, s] = model.NewBoolVar(f"a_{e}_{d}_{g}_{s}")

    # (a) tối đa 1 ca/ngày + chỉ xếp nếu available
    for e in eids:
        for d in range(NUM_DAYS):
            model.Add(sum(assign[e, d, g, s] for g in GATES for s in SHIFT_TYPES) <= 1)
            for g in GATES:
                for s in SHIFT_TYPES:
                    if not availability[e][d][s]:
                        model.Add(assign[e, d, g, s] == 0)

    # (b) Trưởng ca chỉ làm cổng B
    for e in eids:
        if emp_by_id[e].role == ROLE_TC:
            for d in range(NUM_DAYS):
                for g in GATES:
                    if g != "B":
                        for s in SHIFT_TYPES:
                            model.Add(assign[e, d, g, s] == 0)

    # (c) không sáng ngay sau đêm hôm trước
    for e in eids:
        for d in range(NUM_DAYS - 1):
            night_today = sum(assign[e, d, g, "dem"] for g in GATES)
            morning_next = sum(assign[e, d + 1, g, "sang"] for g in GATES)
            model.Add(night_today + morning_next <= 1)

    # (d) khóa các assignment đã approve
    for (e, d), val in locked.items():
        for g in GATES:
            for s in SHIFT_TYPES:
                target = 1 if (val != "OFF" and val == (g, s)) else 0
                model.Add(assign[e, d, g, s] == target)

    # (e) min-staffing + slack (manual review)
    shortfall_vars, lead_shortfall_vars = [], []
    for d in range(NUM_DAYS):
        for g in GATES:
            for s in SHIFT_TYPES:
                req = REQUIREMENTS[g][s]
                total_needed = req.get("NV", 0) + req.get("LEAD", 0)
                people_here = [assign[e, d, g, s] for e in eids]
                if total_needed > 0:
                    short = model.NewIntVar(0, total_needed, f"short_{g}_{d}_{s}")
                    model.Add(sum(people_here) + short >= total_needed)
                    shortfall_vars.append((g, d, s, short))
                if g in LEAD_GATES and req.get("LEAD", 0) > 0:
                    lead_people = [assign[e, d, g, s] for e in eids if emp_by_id[e].role in (ROLE_TC, ROLE_PC)]
                    strict_tc = [assign[e, d, g, s] for e in eids if emp_by_id[e].role == ROLE_TC]
                    lead_short = model.NewIntVar(0, req["LEAD"], f"leadshort_{g}_{d}_{s}")
                    if req.get("lead_mandatory_role"):
                        model.Add(sum(strict_tc) + lead_short >= req["LEAD"])
                    else:
                        model.Add(sum(lead_people) + lead_short >= req["LEAD"])
                    lead_shortfall_vars.append((g, d, s, lead_short))

    # (f) cân bằng effort THEO GIỜ (xét thời lượng ca khác nhau theo cổng), nghỉ phép -> giảm target theo tỷ lệ, không ép bù
    deviation_vars = []
    for e in eids:
        emp = emp_by_id[e]
        available_days = sum(1 for d in range(NUM_DAYS) if d not in emp.leave_days)
        target_hours = round(BASE_TARGET_HOURS_PER_MONTH * available_days / NUM_DAYS)
        total_hours = sum(
            SHIFT_HOURS[g][s] * assign[e, d, g, s]
            for d in range(NUM_DAYS) for g in GATES for s in SHIFT_TYPES
        )
        max_possible_hours = NUM_DAYS * max(SHIFT_HOURS[g][s] for g in GATES for s in SHIFT_TYPES)
        dev = model.NewIntVar(0, max_possible_hours, f"dev_{e}")
        model.AddAbsEquality(dev, total_hours - target_hours)
        deviation_vars.append(dev)

    # (g) ưu tiên xen kẽ: phạt streak 3 ngày liên tiếp cùng loại ca
    streak_vars = []
    for e in eids:
        for d in range(NUM_DAYS - 2):
            for stype in SHIFT_TYPES:
                worked = [model.NewBoolVar(f"w_{e}_{d}_{k}_{stype}") for k in range(3)]
                for k in range(3):
                    model.Add(sum(assign[e, d + k, g, stype] for g in GATES) == worked[k])
                streak = model.NewBoolVar(f"streak_{e}_{d}_{stype}")
                model.AddBoolAnd(worked).OnlyEnforceIf(streak)
                model.AddBoolOr([w.Not() for w in worked]).OnlyEnforceIf(streak.Not())
                streak_vars.append(streak)

    model.Minimize(
        SHORTFALL_PENALTY * sum(v for *_, v in shortfall_vars)
        + LEAD_SHORTFALL_PENALTY * sum(v for *_, v in lead_shortfall_vars)
        + BALANCE_PENALTY_WEIGHT * sum(deviation_vars)
        + STREAK_PENALTY_WEIGHT * sum(streak_vars)
    )

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit_s
    solver.parameters.num_search_workers = 8
    status = solver.Solve(model)

    rows = []
    for e in eids:
        row = {"employee": e, "role": emp_by_id[e].role}
        for d in range(NUM_DAYS):
            cell = "OFF"
            for g in GATES:
                for s in SHIFT_TYPES:
                    if solver.Value(assign[e, d, g, s]):
                        cell = f"{g}-{s}"
            row[DAYS[d].isoformat()] = cell
        rows.append(row)
    schedule_df = pd.DataFrame(rows).set_index("employee")

    shortage_rows = []
    for g, d, s, var in shortfall_vars:
        if solver.Value(var) > 0:
            shortage_rows.append({"date": DAYS[d].isoformat(), "gate": g, "shift": s,
                                   "type": "thiếu NV/tổng", "missing": solver.Value(var)})
    for g, d, s, var in lead_shortfall_vars:
        if solver.Value(var) > 0:
            shortage_rows.append({"date": DAYS[d].isoformat(), "gate": g, "shift": s,
                                   "type": "thiếu trưởng ca/phó ca", "missing": solver.Value(var)})
    shortage_df = pd.DataFrame(shortage_rows)

    stats = {"status": solver.StatusName(status),
             "objective": solver.ObjectiveValue() if status in (cp_model.OPTIMAL, cp_model.FEASIBLE) else None,
             "wall_time_s": round(solver.WallTime(), 2)}
    return status, schedule_df, shortage_df, stats


## 4. Chạy solve cho cả tháng (lần đầu)

Output chính là **bảng lịch dạng excel** (hàng = nhân viên, cột = ngày, giá trị = `Cổng-Ca` hoặc `OFF`) — đúng format quản lý đang quen dùng để dễ migrate.

In [ ]:
status, schedule_df, shortage_df, stats = solve_schedule(employees, availability, time_limit_s=30)
print("Kết quả solve:", stats)
schedule_df


In [ ]:
print(f"Số dòng thiếu hụt cần MANUAL REVIEW: {len(shortage_df)}")
shortage_df


## 5. Kiểm tra cân bằng effort & nghỉ phép

Bảng dưới so sánh **target giờ cho cả horizon 4 tuần** (44h/tuần × 4, đã tự động giảm theo tỷ lệ nếu có nghỉ phép) với **số giờ thực tế được xếp** (tính đúng theo `SHIFT_HOURS` của từng cổng/ca) — để xác nhận cơ chế bù trừ/cân bằng hoạt động đúng, và nhân viên nghỉ phép **không bị ép bù**.

Với target đúng là 44h/**tuần**, và `SHIFT_HOURS` giờ là **số giờ thật** theo từng cổng (ca sáng 11h mọi cổng; ca đêm: A/G 13h, B/D 12h), số liệu mẫu 21 nhân viên gần khớp với nhu cầu (~22-23 người) — actual hours trung bình sẽ nhỉnh hơn target một chút, không còn lệch nặng như các lần tính sai đơn vị trước đó.

In [ ]:
summary_rows = []
for e in employees:
    available_days = sum(1 for d in range(NUM_DAYS) if d not in e.leave_days)
    target_hours = round(BASE_TARGET_HOURS_PER_MONTH * available_days / NUM_DAYS)
    actual_hours = 0
    actual_shifts = 0
    for d in range(NUM_DAYS):
        cell = schedule_df.loc[e.eid, DAYS[d].isoformat()]
        if cell != "OFF":
            g, s = cell.split("-")
            actual_hours += SHIFT_HOURS[g][s]
            actual_shifts += 1
    summary_rows.append({
        "employee": e.eid, "role": e.role, "leave_days": len(e.leave_days),
        "target_hours": target_hours, "actual_hours": actual_hours,
        "deviation_hours": actual_hours - target_hours, "actual_shifts": actual_shifts,
    })
summary_df = pd.DataFrame(summary_rows).set_index("employee")
print(f"Trung bình giờ thực tế: {summary_df['actual_hours'].mean():.0f}h  "
      f"(target: {BASE_TARGET_HOURS_PER_MONTH}h cho {NUM_WEEKS} tuần, tương đương {BASE_TARGET_HOURS_PER_WEEK}h/tuần).")
summary_df


## 6. Demo: sự cố đột xuất + đổi ca CỤC BỘ, không ghi đè lịch đã approve

Mô phỏng đúng nghiệp vụ: *"khi thay đổi không làm ảnh hưởng đến lịch của nhân viên khác"*.

Cách làm: coi **toàn bộ lịch vừa solve ở bước 4 là đã được quản lý approve** → khóa (`locked`) tất cả các ô. Khi có sự cố (nhân viên báo nghỉ ốm đột xuất), ta **chỉ mở khóa đúng ngày bị ảnh hưởng** rồi re-solve — solver sẽ tự tìm người thay thế phù hợp cho riêng ngày đó, còn lại giữ nguyên 100%.

In [ ]:
sick_emp, sick_day = "NV07", 22
print(f"Sự cố: {sick_emp} báo nghỉ ốm đột xuất ngày {DAYS[sick_day].isoformat()}")
print("Ca cũ (trước sự cố):", schedule_df.loc[sick_emp, DAYS[sick_day].isoformat()])

# Khóa toàn bộ lịch đã approve, chỉ mở lại đúng ngày bị ảnh hưởng
locked = {}
for e in schedule_df.index:
    for d in range(NUM_DAYS):
        if d == sick_day:
            continue
        cell = schedule_df.loc[e, DAYS[d].isoformat()]
        locked[(e, d)] = "OFF" if cell == "OFF" else tuple(cell.split("-"))

# Cập nhật availability: nhân viên ốm không còn available ngày đó
availability2 = {e: {d: dict(s) for d, s in day.items()} for e, day in availability.items()}
availability2[sick_emp][sick_day] = {"sang": False, "dem": False}

status2, schedule_df2, shortage_df2, stats2 = solve_schedule(employees, availability2, locked=locked, time_limit_s=20)
print("\nKết quả re-solve cục bộ:", stats2)


In [ ]:
# Kiểm tra: chỉ ngày bị ảnh hưởng thay đổi, các ngày khác giữ nguyên 100%
changed = []
for e in schedule_df.index:
    for d in range(NUM_DAYS):
        col = DAYS[d].isoformat()
        old, new = schedule_df.loc[e, col], schedule_df2.loc[e, col]
        if old != new:
            changed.append((e, col, old, new))

sick_date_str = DAYS[sick_day].isoformat()
leaked = [c for c in changed if c[1] != sick_date_str]

print(f"Tổng số ô thay đổi: {len(changed)}")
print(f"Số ô thay đổi RA NGOÀI ngày bị ảnh hưởng (phải = 0): {len(leaked)}")
pd.DataFrame(changed, columns=["employee", "date", "old", "new"])


## 7. Kết luận & hạn chế của MVP

**Đã hiện thực & verify được:**
- Solver ra lịch thỏa toàn bộ các constraint cứng (max 1 ca/ngày, TC chỉ ở cổng B, không sáng-sau-đêm, giữ lịch đã approve).
- Cơ chế "thiếu người → cờ manual review" thay vì solver bị treo.
- Cân bằng effort **tính đúng theo giờ** (có xét thời lượng ca khác nhau theo từng cổng), có tính đến nghỉ phép (không ép bù).
- Đổi ca cục bộ khi có sự cố, xác nhận bằng test **0 ô thay đổi ngoài phạm vi ảnh hưởng**.

**Phát hiện quan trọng đã được xác nhận lại (44h/TUẦN, không phải /tháng):**
**Cập nhật với số giờ ca THẬT (từ bảng phân ca công ty cung cấp):** ca sáng 11h ở mọi cổng; ca đêm: A/G 13h, B/D 12h. Với target đúng **44h/tuần** (~176h cho horizon 4 tuần), tổng nhu cầu tối thiểu 4 cổng ≈ **140 giờ-người/ngày** (~3,920 giờ-người/tháng) → cần tối thiểu **~22-23 nhân viên**. Bộ dữ liệu mẫu có 21 người nên khá sát — actual hours trung bình cao hơn target một chút (~192h so với target ~174h, tức nhỉnh hơn ~10%), hợp lý và không còn là dấu hiệu thiếu hụt nghiêm trọng như các lần tính sai đơn vị/số liệu trước.

**Chưa nằm trong MVP (cần làm ở vòng sau):**
- UI thật cho nhân viên đăng ký ca / quản lý review-approve (hiện đang dùng data mẫu hardcode).
- Xử lý nhiều yêu cầu đổi ca xảy ra đồng thời (concurrency).
- Wrap thành FastAPI service + tích hợp backend Golang như trong tài liệu gốc.
- Lưu trữ lịch sử approve/audit trail.
- Ràng buộc nghỉ ngơi tối thiểu theo GIỜ thực tế giữa 2 ca bất kỳ (hiện giữ quy tắc đơn giản "không sáng ngay sau đêm" theo yêu cầu).
- Tinh chỉnh trọng số objective (`SHORTFALL_PENALTY`, `BALANCE_PENALTY_WEIGHT`,...) dựa trên phản hồi thực tế của quản lý.